# Loading dataset and model from Mlflow 

In [8]:
import pandas as pd 
import mlflow
import mlflow.sklearn

RUN_ID = "a084293413704d02967935b576fb8f5c"

MODEL_URI = f"runs:/{RUN_ID}/model"

mlflow.set_tracking_uri("file:///D:/projects/home_default/mlruns/")
model = mlflow.sklearn.load_model(MODEL_URI)

df = pd.read_csv('../Dataset/Loan_default.csv')

In [9]:
df.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='str')

In [10]:
print(model)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num_scaled', 'passthrough',
                                                  ['Income', 'LoanAmount',
                                                   'CreditScore']),
                                                 ('num_unscaled', 'passthrough',
                                                  ['Age', 'MonthsEmployed',
                                                   'NumCreditLines',
                                                   'LoanTerm']),
                                                 ('binary',
                                                  OrdinalEncoder(categories=[['No',
                                                                              'Yes'],
                                                                             ['No',
                                                                              'Yes'],
                                                        

# spliting data 

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop("Default", axis=1)
y = df["Default"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp  
)


# Finding probabilities for validation set 

In [12]:
pd_scores = model.predict_proba(X_val)[:,1]

d:\projects\home_default\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# FP and FN cost calculations

In [13]:
# risk for default
avg_risk = df['LoanAmount'].mean()
# gain for when not defaulted
gain = df['LoanAmount'] * df['InterestRate'] * (df['LoanTerm'] / 12)

lgd = 0.6 



# Threshold calculations

In [14]:
from sklearn.metrics import confusion_matrix
import numpy as np 

thresholds = np.arange(0.05, 0.95, 0.01)
results = []
for t in thresholds:
    y_pred = (pd_scores > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    total_loss = (fn * avg_risk * lgd) + (fp * gain.mean())

    approval_rate = (tn + tp) / len(y_val)
    default_rate_approved = tp / (tp + fp) if (tp + fp) > 0 else 0
    results.append({
        "threshold": t,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "tp": tp,
        "total_loss": total_loss,
        "approval_rate": approval_rate,
        "default_rate_approved": default_rate_approved
    })

In [15]:
df_results = pd.DataFrame(results)

best_row = df_results.loc[df_results["total_loss"].idxmin()]

best_threshold = best_row["threshold"]

print(f"Best business threshold: {best_threshold}")
print(f"Minimum loss: {best_row['total_loss']:.2f}")
print(f"Approval rate: {best_row['approval_rate']:.2%}")
print(f"Default rate among approved: {best_row['default_rate_approved']:.2%}")

Best business threshold: 0.8500000000000002
Minimum loss: 227039349.07
Approval rate: 88.38%
Default rate among approved: 0.00%


# Summary of Findings

## Business Cutoff Threshold Analysis

This notebook implements a **cost-sensitive threshold optimization** for loan default prediction to maximize business value.

### Methodology

1. **Model**: Loaded pre-trained ML model from MLflow (Run ID: a084293413704d02967935b576fb8f5c)
2. **Data Split**: 60% training, 20% validation, 20% test set (stratified by target)
3. **Cost Calculations**:
   - **FN Cost** (False Negatives): Loan defaults = `fn × avg_loan_amount × LGD (0.6)`
   - **FP Cost** (False Positives): Lost interest income = `fp × (loan_amount × interest_rate × loan_term/12)`
4. **Optimization**: Identified threshold minimizing total financial loss across validation set

### Key Results

- **Optimal Threshold**: Probability cutoff that maximizes net business value
- **Total Business Loss**: Monetary impact of both missed defaults and rejected good customers
- **Approval Rate**: Percentage of loan applications approved at optimal threshold
- **Default Rate Among Approved**: Quality of approved portfolio (lower is better)

### Interpretation

The optimal threshold balances:
- **Precision vs Recall**: Not just accuracy, but actual business costs
- **Portfolio Quality**: Maintaining acceptable default rate among approved loans
- **Revenue**: Approving profitable customers while avoiding high-loss defaults

This approach ensures the model decision boundary aligns with business objectives rather than pure predictive accuracy.